# Feature Selection and Parametrics

**ACP · Analytical Community of Practice — July 2026 · Josh**

> *"Deciding which features matter is one disciplined, statistical process — and it's the **same** whether your features are TF-IDF columns or BERT embeddings."*

**How this notebook is used on stage:** pre-built and rehearsed — walked live, with select cells re-run. No improvised live-coding. Before the talk: **Kernel ▸ Restart Kernel and Run All Cells** — must run clean, top to bottom.

**Env:** local Python (Anaconda); exact versions print in §0. Local data only (`../data/cpsc_merged.csv`) — never the lake, no cloud, no BERT in the demo.

---
## ✂️ Formatting cheatsheet — *delete this whole section before the talk*

### Math
MathJax, same syntax as the Obsidian briefings (no space just inside the `$`):

- inline: `$\lambda$` → $\lambda$, `$\hat\beta_j$` → $\hat\beta_j$
- display, on its own lines:

$$\hat{\beta}^{\text{lasso}} = \arg\min_{\beta}\ \underbrace{\lVert y - X\beta\rVert_2^2}_{\text{fit}} + \underbrace{\lambda\lVert\beta\rVert_1}_{\text{penalty}}$$

- multi-line derivations with `aligned`:

$$\begin{aligned}
\chi^2 &= \sum_i \frac{(O_i - E_i)^2}{E_i} \\
E_i &= \frac{\text{row}_i \times \text{col}_i}{N}
\end{aligned}$$

### Images
Paths are relative to `notebooks/`:

- full width: `![L1 vs L2](../figures/wb_14_l1_l2_geometry.png)`
- controlled width (use this on stage): `<img src="../figures/wb_14_l1_l2_geometry.png" width="550"/>`

<img src="../figures/wb_14_l1_l2_geometry.png" width="550"/>

### Tables

| method | type | cost |
|---|---|---|
| $\chi^2$ / MI / F | filter | cheap, model-free |
| L1, tree importances | embedded | one fit |
| RFE | wrapper | many fits |

### Callouts
Obsidian's `> [!note]` renders as a plain blockquote in Jupyter (the `[!note]` shows literally). For anything shown on stage use:

> **⚠️ Note:** like this.

In [ ]:
# ✂️ HOW-TO — the three output patterns this talk uses (self-contained; delete with this section)
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Math, display

# 1) Never assert what you can show — print the actual vectors, small enough for a slide
v_fire  = np.array([0.61, 0.02, 0.33])
v_flame = np.array([0.58, 0.05, 0.31])
cos = float(v_fire @ v_flame / (np.linalg.norm(v_fire) * np.linalg.norm(v_flame)))
print("fire  :", v_fire)
print("flame :", v_flame)
print(f"cosine similarity = {cos:.3f}")

# 2) Render a *computed* number as LaTeX — no retyped values on slides
lam = 0.10
display(Math(rf"\lambda_{{\text{{chosen}}}} = {lam:.2f}"))

# 3) Figures: back-row-legible, saved to ../figures/ beside the code that made them
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([0, 1, 2, 3], [3.1, 2.2, 1.8, 1.7], marker="o", lw=3)
ax.set(xlabel="epoch", ylabel="loss", title="example — delete me")
fig.tight_layout()
# fig.savefig("../figures/example.png", bbox_inches="tight")   # uncomment to keep
plt.show()

---
## 0 · Setup & environment — run before anything else

In [ ]:
# 0 · versions (env pin check), seed, back-row-legible plot defaults, data path
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    import sklearn
    SKLEARN = sklearn.__version__
except ImportError:
    SKLEARN = "MISSING — §3 (the demo) requires scikit-learn"

print(f"python     {platform.python_version()}")
print(f"numpy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"sklearn    {SKLEARN}")

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "font.size": 16,        # legible from the back row
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "figure.dpi": 110,
})

FIGURES = Path("../figures")
DATA = Path("../data/cpsc_merged.csv")
print(f"\ndata present: {DATA.exists()}  ({DATA})")

---
## 1 · Latent space — "you already trained one" (~8 min)

> **⚠️ Intuition only — say so on stage.** This is orientation, not derivation.

**Beats:**

- Hook: the embedding line in `gpt_train.py` — *about half the room ran this; open with a 30-s recap for the other half*
- Words as vectors → geometry = meaning → nearest neighbors → one analogy
- The "base model" beat: next-char prediction → why the output is semi-gibberish
- *"This is BERT's family — hold that thought"* (pays off in §4.2)

**TODO:** pull loss curve + generated samples from the one instrumented re-run (`gpt_train_instrumented.py`).

<img src="../figures/wb_10_loss_curve.png" width="520"/>

In [ ]:
# 1 · TODO: embedding walk-through — real vectors + nearest neighbors
# (lift from briefing-computations.ipynb; PRINT the actual vectors — never assert)

---
## 2 · Feature selection + parametrics (~12 min)

The statistical methods that say **which features matter**:

- **2.1 Filter** — $\chi^2$, mutual information, F-score (cheap, model-free)
- **2.2 Embedded** — L1/lasso coefficients, tree importances (one fit)
- **2.3 Wrapper/RFE** — why it's costly (many fits — mention, don't run)

> **The invariant (the talk's spine):** identical scores apply to TF-IDF columns *and* embedding dimensions.

Figures ready: `wb_12_chi2.png` · `wb_13_selection_knee.png` · `wb_14_l1_l2_geometry.png` · `wb_16_l1_path.png`

In [ ]:
# 2 · TODO: chi2 contingency on one word ('fire') + all-vocab scores on CPSC
# (worked numbers exist in acp-workbook.ipynb FIG 12/13 cells)

---
## 3 · The demo — one connected pipeline (~20 min · the centerpiece)

Local CPSC + classical models only. Rehearse which cells re-run live.

**text → TF-IDF → select → tune (CV) → several model *types* → pick on evidence**

### 3.1 Load the local data

In [ ]:
# 3.1 · local data only — reproducible on any colleague's machine
df = pd.read_csv(DATA)
print(df.shape)
df.head(3)

### 3.2 Text → TF-IDF

In [ ]:
# 3.2 · TODO: TfidfVectorizer (current sklearn idiom — note what changed vs. older habits)
# from sklearn.feature_extraction.text import TfidfVectorizer

### 3.3 Select the features that matter

In [ ]:
# 3.3 · TODO: SelectKBest(chi2) — INSIDE a Pipeline, so selection lives inside CV
# (this is the leakage rule; it pays off in §4.1)
# from sklearn.pipeline import Pipeline
# from sklearn.feature_selection import SelectKBest, chi2

### 3.4 Tune with cross-validation

In [ ]:
# 3.4 · TODO: RandomizedSearchCV / GridSearchCV over the pipeline; scoring='f1_macro'

### 3.5 The bake-off — several model *types*, identical folds

In [ ]:
# 3.5 · TODO: logistic / LinearSVC / ComplementNB / tree — same folds, macro-F1
# (working version: acp-workbook.ipynb bake-off cell)

### 3.6 Pick on evidence

CV macro-F1 table → the choice justifies itself. Figure ready: `wb_19_bakeoff.png`

In [ ]:
# 3.6 · TODO: final comparison table (mean ± sd across folds), winner highlighted

---
## 4 · Close (~5 min)

### 4.1 The leakage gotcha
A model that looks great — and isn't: select-then-CV vs Pipeline-in-CV. Figure ready: `wb_20_leakage.png`

In [ ]:
# 4.1 · TODO: leakage demo on pure-noise labels (working version: acp-workbook.ipynb)

### 4.2 The BERT punchline — **CITED, not recomputed**

> **⚠️ Attribution rule:** these numbers come from [colleague]'s production BERT work. Label them as cited on the slide; never recompute, adjust, or extend them here.

**TODO:** classical plateau number → BERT gain number → callback to §1 (*embeddings capture relationships TF-IDF can't*).

---
## Appendix · depth markers & Q&A parking lot

Backup material for the PhD end of the room — let Q&A absorb the deep end.

- TODO: Ridge-vs-Lasso constraint-geometry derivation (backup)
- TODO: bias–variance decomposition (backup)
- Q&A parking lot — fill during rehearsal:

---
**Reproducibility:** every figure and number shown traces to code in this repo — *except* the §4.2 BERT numbers, which are cited from a colleague's production work and labeled as such.